# Lab 08: Vision & Sensory Delay
**Computational Sensorimotor Control — Week 8**

**Task:** Build the Sensor class (delay buffer + noise injection), then reproduce four key figures from the lecture.

**Key concepts:**
- Sensory delay means feedback is always stale: y(t) = x(t − Δ) + ε(t)
- Two visual channels: peripheral (gaze on target, Δ=80ms, σ=5mm) and central (gaze tracks hand, Δ=150ms, σ=1mm)
- PD correction works when sensor noise is small (Figure 2, σ=0.3mm); P-only on the arc (σ=5mm) because differentiating noisy position amplifies noise

**Figures reproduced:** 5 (Sensor verification), 2 (delay instability), 7 (three-way arc comparison), 9 (updated frontier)


In [ ]:
!pip3 install git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from matplotlib.patches import ConnectionPatch
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11,
                      'axes.grid': True, 'grid.alpha': 0.3})

from smc import (
    Arm, Q_REF,
    mass_matrix, coriolis, joint_accelerations,
    Sensor,
)

arm = Arm()
ik  = arm.inverse_kinematics
fk  = arm.forward_kinematics
Jfn = arm.jacobian
start_hand = np.array(fk(Q_REF))
dt = 0.001
k_noise = 0.15

def minjerk_basis(t, T):
    tau = np.clip(t / T, 0, 1)
    return 10*tau**3 - 15*tau**4 + 6*tau**5

def minjerk_reach(target, T, dt=0.001):
    t = np.arange(0, T + dt/2, dt)
    s = minjerk_basis(t, T)
    return t, start_hand[None,:] + s[:,None] * (target - start_hand)[None,:]

def minjerk_arc(R, T, dt=0.001):
    center = start_hand + np.array([R, 0])
    t = np.arange(0, T + dt/2, dt)
    s = minjerk_basis(t, T)
    th = np.pi * (1 - s)
    return t, np.column_stack([center[0]+R*np.cos(th), center[1]+R*np.sin(th)])

def dense_arc(R, n=500):
    center = start_hand + np.array([R, 0])
    th = np.linspace(np.pi, 0, n)
    return np.column_stack([center[0]+R*np.cos(th), center[1]+R*np.sin(th)])

def id_torques(t_arr, pos):
    n = len(t_arr)
    q = np.array([ik(pos[i,0], pos[i,1]) for i in range(n)])
    qd = np.gradient(q, dt, axis=0); qdd = np.gradient(qd, dt, axis=0)
    tau = np.zeros((n, 2))
    for i in range(n): tau[i] = mass_matrix(q[i]) @ qdd[i] + coriolis(q[i], qd[i])
    return q, qd, qdd, tau

def add_sdn(tau, k, rng):
    return tau + k * np.abs(tau) * rng.standard_normal(2)

NAVY='#1B2A4A'; TEAL='#2E86AB'; RED='#E74C3C'; GREEN='#27AE60'; GRAY='#7F8C8D'; ORANGE='#E67E22'
print('Setup complete.')


---
## Part 1: Implement & Verify the Sensor Class (Lecture §3 — Figure 5)

A `Sensor` models a sensory channel: **y(t) = x(t − Δ) + ε(t)**.

**Your task:** Fill in the two lines marked `# YOUR CODE HERE`.


In [ ]:
from math import ceil

class MySensor:
    def __init__(self, delay, R, dt=0.001):
        self.delay = delay
        self.R = np.asarray(R, dtype=float)
        self.dt = dt
        self.buf_len = max(1, ceil(delay / dt))
        self._L = np.linalg.cholesky(self.R)
        self.reset()

    def reset(self):
        self._buf = []

    def sense(self, true_state, rng):
        true_state = np.asarray(true_state, dtype=float)
        self._buf.append(true_state.copy())
        # 1. Read delayed state: index = max(0, len(buf) - 1 - buf_len)
        delayed_state = ...  # YOUR CODE HERE (one line)
        # 2. Add noise: z ~ N(0,I), noise = L @ z
        noise = ...  # YOUR CODE HERE (one line)
        return delayed_state + noise

print('MySensor class defined.')


In [ ]:
# ── Delay verification ──
s_test = MySensor(delay=0.150, R=np.eye(2)*1e-20)
s_test.reset(); rng_test = np.random.default_rng(0)
outputs = []
for i in range(300):
    x = np.array([0.0, 0.0]) if i < 100 else np.array([1.0, 1.0])
    outputs.append(s_test.sense(x, rng_test).copy())
outputs = np.array(outputs)
print(f't=249: y[0] = {outputs[249,0]:.4f}  (should be ~0)')
print(f't=250: y[0] = {outputs[250,0]:.4f}  (should be ~1)')
assert outputs[249,0] < 0.5 and outputs[250,0] > 0.5
print('\n✓ Delay verification passed.')


In [ ]:
# ── Compare MySensor vs smc.Sensor ──
_, pos_cmp = minjerk_reach(start_hand + np.array([0.10, 0]), 0.3)

# Run both on the same trajectory with the same seed
mine = MySensor(delay=0.080, R=np.eye(2)*0.005**2)
ref  = Sensor(delay=0.080, R=np.eye(2)*0.005**2)
mine.reset(); ref.reset()
rng1 = np.random.default_rng(42); rng2 = np.random.default_rng(42)

ym_all = []; yr_all = []
for i in range(len(pos_cmp)):
    ym_all.append(mine.sense(pos_cmp[i], rng1))
    yr_all.append(ref.sense(pos_cmp[i], rng2))
ym_all = np.array(ym_all); yr_all = np.array(yr_all)
max_err = np.max(np.abs(ym_all - yr_all))

# Print side-by-side at a few timesteps
print(f'{"t (ms)":>7} | {"MySensor x":>12} | {"smc.Sensor x":>12} | {"diff":>10}')
print('-' * 52)
for t_idx in [100, 150, 200, 250]:
    print(f'{t_idx:7d} | {ym_all[t_idx,0]*100:12.6f} | {yr_all[t_idx,0]*100:12.6f} | {abs(ym_all[t_idx,0]-yr_all[t_idx,0]):.2e}')

print(f'\nMax difference across {len(pos_cmp)} timesteps: {max_err:.2e}')
assert max_err < 1e-12, f'Mismatch: {max_err}'
print('✓ Your MySensor matches smc.Sensor exactly.')


Now reproduce **Figure 5**.

**Your task:** Fill in the delay and covariance for each sensor.


In [ ]:
tgt=start_hand+np.array([0.10,0]); T=0.6
t_arr,pos=minjerk_reach(tgt,T)
central = MySensor(delay=..., R=...)  # YOUR CODE HERE
periph  = MySensor(delay=..., R=...)  # YOUR CODE HERE

rng = np.random.default_rng(77)
central.reset(); periph.reset()
mc=[]; mp=[]
for i in range(len(t_arr)):
    mc.append(central.sense(pos[i], rng)); mp.append(periph.sense(pos[i], rng))
mc=np.array(mc); mp=np.array(mp)
skip_c=int(0.150/dt)+10; skip_p=int(0.080/dt)+10
noise_c=np.zeros_like(mc); noise_p=np.zeros_like(mp)
for i in range(len(t_arr)):
    noise_c[i]=mc[i]-pos[max(0,i-int(0.150/dt))]
    noise_p[i]=mp[i]-pos[max(0,i-int(0.080/dt))]
fig,(a1,a2)=plt.subplots(1,2,figsize=(13,5))
t_ms=t_arr*1000; mask=(t_ms>=150)&(t_ms<=450)
a1.plot(t_ms[mask],pos[mask,0]*100,'-',color=GRAY,lw=3,label='True hand x(t)',zorder=5)
a1.plot(t_ms[mask],mc[mask,0]*100,'-',color=TEAL,lw=1.2,alpha=0.9,
        label='Central (Δ=150ms, σ=1mm) — gaze tracks hand')
a1.plot(t_ms[mask],mp[mask,0]*100,'-',color=ORANGE,lw=1.2,alpha=0.9,
        label='Peripheral (Δ=80ms, σ=5mm) — gaze on target')
a1.set_xlabel('Time (ms)'); a1.set_ylabel('x position (cm)')
a1.set_title('A: Delay Verification',fontweight='bold',loc='left')
a1.legend(fontsize=8,loc='lower right')
bins=np.arange(-2.0,2.05,0.05)
a2.hist(noise_c[skip_c:,0]*100,bins=bins,alpha=0.7,color=TEAL,density=True,
        label=f'Central (SD={np.std(noise_c[skip_c:,0])*100:.2f} cm)')
a2.hist(noise_p[skip_p:,0]*100,bins=bins,alpha=0.5,color=ORANGE,density=True,
        label=f'Peripheral (SD={np.std(noise_p[skip_p:,0])*100:.2f} cm)')
xx=np.linspace(-2,2,200)
a2.plot(xx,norm.pdf(xx,0,0.1),'--',color=TEAL,lw=2,label='Expected N(0,1mm)')
a2.plot(xx,norm.pdf(xx,0,0.5),'--',color=ORANGE,lw=2,label='Expected N(0,5mm)')
a2.set_xlabel('Noise in x (cm)'); a2.set_ylabel('Density')
a2.set_title('B: Noise Distribution',fontweight='bold',loc='left')
a2.legend(fontsize=8); a2.set_xlim(-2,2)
plt.tight_layout(); plt.show()
print(f'Central noise SD:    {np.std(noise_c[skip_c:,0])*1000:.2f} mm (expected: 1.00)')
print(f'Peripheral noise SD: {np.std(noise_p[skip_p:,0])*1000:.2f} mm (expected: 5.00)')


---
## Part 2: The Cost of Delay (Lecture §1 — Figure 2)

A feedforward + PD controller with velocity error:

**τ = τ_ff + J^T · (Kp · e_p + Kd · e_v)**

Parameters: Kp=5, Kd=1.5, σ=0.3mm, 10 cm reach, T=600ms.
At Δ=0 the correction is synchronized — paths track the desired trajectory.
At Δ=200ms the stale correction overshoots and destabilizes.

**Your task:** Fill in the one line for the PD correction torque.


In [ ]:
from math import ceil as _ceil

class _Sensor:
    """Lightweight sensor (same logic as MySensor)."""
    def __init__(s,delay,sigma): s.bl=max(1,_ceil(delay/dt)); s.sigma=sigma; s.buf=[]
    def reset(s): s.buf=[]
    def sense(s,x,rng):
        s.buf.append(x.copy()); idx=max(0,len(s.buf)-1-s.bl)
        return s.buf[idx]+rng.normal(0,s.sigma,len(x))

def sim_delay_pd(q0, tau_ff, pos_des, vel_des, Kp, Kd, delay_ms, sigma, rng):
    """FF + PD correction with delay and velocity error."""
    sen = _Sensor(delay_ms/1000, sigma); sen.reset()
    n = len(tau_ff)
    q=np.zeros((n,2)); qd=np.zeros((n,2)); hand=np.zeros((n,2))
    q[0]=q0.copy(); hand[0]=fk(q0); prev_y=hand[0].copy()
    for i in range(n-1):
        y = sen.sense(hand[i], rng)
        yd = (y - prev_y)/dt; prev_y = y.copy()
        ep = pos_des[min(i,len(pos_des)-1)] - y
        ev = vel_des[min(i,len(vel_des)-1)] - yd
        # PD correction: J^T * (Kp*ep + Kd*ev)
        tc = ...  # YOUR CODE HERE (one line)
        tt = tau_ff[i] + tc
        qdd = joint_accelerations(q[i], qd[i], tt)
        q[i+1]=q[i]+qd[i]*dt+0.5*qdd*dt**2
        qd[i+1]=qd[i]+qdd*dt; hand[i+1]=fk(q[i+1])
    return hand
print('sim_delay_pd defined.')


Now reproduce **Figure 2**. Fill in delay values (seconds).


In [ ]:
# ── Reproduce Figure 2 ──
tgt2 = start_hand + np.array([0.10, 0]); T2 = 0.6
t2, pos2 = minjerk_reach(tgt2, T2)
vel2 = np.gradient(pos2, dt, axis=0)
q2, _, _, tau2 = id_torques(t2, pos2)

# Three delay conditions — fill in delay values (in milliseconds)
conditions = [
    (..., GREEN, r'$\Delta$ = 0 ms'),    # YOUR CODE HERE
    (..., TEAL,  r'$\Delta$ = 100 ms'),  # YOUR CODE HERE
    (..., RED,   r'$\Delta$ = 200 ms'),  # YOUR CODE HERE
]

fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(16, 4.5))
for dms, col, lab in conditions:
    hands = []
    for trial in range(15):
        rng = np.random.default_rng(100 + trial)
        h = sim_delay_pd(q2[0], tau2, pos2, vel2, 5, 1.5, dms, 0.0003, rng)
        hands.append(h)
    for h in hands: a1.plot(h[:,0]*100, h[:,1]*100, color=col, alpha=0.35, lw=0.8)
    a1.plot([], [], color=col, lw=2, label=lab)
    a2.plot(t2*1000, hands[0][:,0]*100, color=col, lw=1.5, label=lab)
    a3.plot(t2*1000, hands[0][:,1]*100, color=col, lw=1.5, label=lab)

a1.plot(pos2[:,0]*100, pos2[:,1]*100, '--', color=GRAY, lw=2, label='Desired', zorder=1)
a1.plot(start_hand[0]*100, start_hand[1]*100, 'D', color=NAVY, ms=8, zorder=10)
a1.set_xlabel('x (cm)'); a1.set_ylabel('y (cm)'); a1.legend(fontsize=8)
a1.set_title('A: Hand Paths (15 trials)', fontweight='bold', loc='left')

a2.plot(t2*1000, pos2[:,0]*100, '--', color=GRAY, lw=2, label='Desired')
a2.set_xlabel('Time (ms)'); a2.set_ylabel('x (cm)')
a2.set_title('B: x(t), single trial', fontweight='bold', loc='left'); a2.legend(fontsize=8)

a3.plot(t2*1000, pos2[:,1]*100, '--', color=GRAY, lw=2, label='Desired')
a3.set_xlabel('Time (ms)'); a3.set_ylabel('y (cm)')
a3.set_title('C: y(t), single trial', fontweight='bold', loc='left'); a3.legend(fontsize=8)
plt.tight_layout(); plt.show()

print('At Δ=0, feedback is synchronized — paths hug the desired trajectory.')
print('At Δ=200ms, stale correction overshoots and destabilizes x and y.')


---
## Part 3: Three-Way Comparison on the Arc (Lecture §4 — Figure 7)

Compare open-loop, peripheral (gaze on target: Δ=80ms, σ=5mm), and central (gaze tracks hand: Δ=150ms, σ=1mm)
on the 6 cm arc (T=800ms). **P-only** (Kp=8, no Kd).

**Your task:** Fill in delay and sigma for each feedback condition.


In [ ]:
def sim_delay_p(q0, tau_ff, pos_des, Kp, delay_s, sigma, rng):
    n=len(tau_ff); q=np.zeros((n,2)); qd=np.zeros((n,2)); hand=np.zeros((n,2))
    q[0]=q0.copy(); hand[0]=fk(q0); buf=[]; bl=int(delay_s/dt)
    for i in range(n-1):
        buf.append(hand[i].copy()); tc=np.zeros(2)
        if Kp>0 and len(buf)>bl:
            y=buf[-bl-1]+rng.normal(0,sigma,2)
            tc=Jfn(q[i]).T@(Kp*(pos_des[max(0,i-bl)]-y))
        tt=add_sdn(tau_ff[i]+tc, k_noise, rng)
        qdd=joint_accelerations(q[i],qd[i],tt)
        q[i+1]=q[i]+qd[i]*dt+0.5*qdd*dt**2
        qd[i+1]=qd[i]+qdd*dt; hand[i+1]=fk(q[i+1])
    return hand
print('sim_delay_p defined.')


In [ ]:
R_ARC=0.06; T_ARC=0.8; Kp=8; N=200
t_a,pos_a=minjerk_arc(R_ARC,T_ARC)
q_a,_,_,tau_a=id_torques(t_a,pos_a)
des=dense_arc(R_ARC); tgt_a=pos_a[-1]

rng=np.random.default_rng(42)
eps_ol=[]; paths_ol=[]
for trial in range(N):
    h=sim_delay_p(q_a[0],tau_a,pos_a,0,0,0,rng)
    eps_ol.append(h[-1])
    if trial<30: paths_ol.append(h)
eps_ol=np.array(eps_ol)

eps_pv=[]; paths_pv=[]
for trial in range(N):
    h=sim_delay_p(q_a[0],tau_a,pos_a,Kp,...,...,rng)  # YOUR CODE HERE
    eps_pv.append(h[-1])
    if trial<30: paths_pv.append(h)
eps_pv=np.array(eps_pv)

eps_cv=[]; paths_cv=[]
for trial in range(N):
    h=sim_delay_p(q_a[0],tau_a,pos_a,Kp,...,...,rng)  # YOUR CODE HERE
    eps_cv.append(h[-1])
    if trial<30: paths_cv.append(h)
eps_cv=np.array(eps_cv)

sd_ol=np.std(np.linalg.norm(eps_ol-tgt_a,axis=1))*100
sd_pv=np.std(np.linalg.norm(eps_pv-tgt_a,axis=1))*100
sd_cv=np.std(np.linalg.norm(eps_cv-tgt_a,axis=1))*100
print(f'Open-loop:                    SD = {sd_ol:.3f} cm')
print(f'Peripheral (gaze on target):  SD = {sd_pv:.3f} cm ({(1-sd_pv/sd_ol)*100:.0f}%)')
print(f'Central (gaze tracks hand):   SD = {sd_cv:.3f} cm ({(1-sd_cv/sd_ol)*100:.0f}%)')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
# Panel A: full arc + zoomed inset near endpoint
for h in paths_ol[:30]: axes[0].plot(h[:,0]*100,h[:,1]*100,color=RED,alpha=0.12,lw=0.5)
for h in paths_pv[:30]: axes[0].plot(h[:,0]*100,h[:,1]*100,color=ORANGE,alpha=0.12,lw=0.5)
for h in paths_cv[:30]: axes[0].plot(h[:,0]*100,h[:,1]*100,color=TEAL,alpha=0.12,lw=0.5)
axes[0].plot(des[:,0]*100,des[:,1]*100,':',color=GRAY,lw=2,zorder=1)
axes[0].plot([],[],color=RED,lw=2,label='Open-loop')
axes[0].plot([],[],color=ORANGE,lw=2,label='Peripheral (gaze on target)')
axes[0].plot([],[],color=TEAL,lw=2,label='Central (gaze tracks hand)')
axes[0].plot(start_hand[0]*100,start_hand[1]*100,'D',color=NAVY,ms=8,zorder=10)
axes[0].legend(fontsize=7); axes[0].set_xlabel('x (cm)'); axes[0].set_ylabel('y (cm)')
axes[0].set_title('A: Hand Paths',fontweight='bold',loc='left')

# Zoomed inset near endpoint
ep_x = tgt_a[0]*100; ep_y = tgt_a[1]*100
inset = axes[0].inset_axes([0.45, 0.05, 0.50, 0.50])  # [x, y, w, h] in axes coords
for h in paths_ol[:30]: inset.plot(h[:,0]*100,h[:,1]*100,color=RED,alpha=0.25,lw=0.6)
for h in paths_pv[:30]: inset.plot(h[:,0]*100,h[:,1]*100,color=ORANGE,alpha=0.25,lw=0.6)
for h in paths_cv[:30]: inset.plot(h[:,0]*100,h[:,1]*100,color=TEAL,alpha=0.25,lw=0.6)
inset.plot(des[:,0]*100,des[:,1]*100,':',color=GRAY,lw=1.5)
pad = 0.6
inset.set_xlim(ep_x-pad, ep_x+pad); inset.set_ylim(ep_y-pad, ep_y+pad)
inset.set_title('Endpoint zoom', fontsize=8)
inset.tick_params(labelsize=7)
axes[0].indicate_inset_zoom(inset, edgecolor='black', alpha=0.5)

# Panel B: scatter
axes[1].scatter(eps_ol[:,0]*100,eps_ol[:,1]*100,s=10,alpha=0.3,color=RED,label=f'OL ({sd_ol:.2f})')
axes[1].scatter(eps_pv[:,0]*100,eps_pv[:,1]*100,s=10,alpha=0.3,color=ORANGE,label=f'PV ({sd_pv:.2f})')
axes[1].scatter(eps_cv[:,0]*100,eps_cv[:,1]*100,s=10,alpha=0.3,color=TEAL,label=f'CV ({sd_cv:.2f})')
axes[1].plot(tgt_a[0]*100,tgt_a[1]*100,'x',color=NAVY,ms=12,mew=3,zorder=10)
axes[1].legend(fontsize=9); axes[1].set_xlabel('x (cm)'); axes[1].set_ylabel('y (cm)')
axes[1].set_title('B: Endpoint Scatter',fontweight='bold',loc='left')

# Panel C: bar
bars=axes[2].bar(['Open-\nloop','Periph.\n(target)','Central\n(hand)'],
    [sd_ol,sd_pv,sd_cv],color=[RED,ORANGE,TEAL],width=0.5,alpha=0.8)
for b,v in zip(bars,[sd_ol,sd_pv,sd_cv]):
    axes[2].text(b.get_x()+b.get_width()/2,b.get_height()+0.003,f'{v:.2f}',
                 ha='center',fontsize=11,fontweight='bold')
axes[2].set_ylabel('Endpoint SD (cm)')
axes[2].set_title('C: Endpoint SD',fontweight='bold',loc='left')
plt.tight_layout(); plt.show()
print('Peripheral outperforms central — speed beats precision for continuous correction.')


---
## Part 4: The Updated Frontier (Lecture §6 — Figure 9)

Sweep T from 250–1000 ms. Same Kp=8, P-only.

**Your task:** Fill in the `sim_delay_p` call for central and peripheral.

**Look for two crossovers:** (1) ~250 ms — visual correction begins, (2) ~700 ms — central overtakes peripheral.


In [ ]:
T_vals=[0.25,0.30,0.40,0.50,0.60,0.80,1.0]
tgt9=start_hand+np.array([0.10,0]); Kp=8; N9=300
sds_ol=[]; sds_cv=[]; sds_pv=[]
for T in T_vals:
    t9,pos9=minjerk_reach(tgt9,T); q9,_,_,tau9=id_torques(t9,pos9)
    rng=np.random.default_rng(42)
    sds_ol.append(np.std(np.linalg.norm(np.array(
        [sim_delay_p(q9[0],tau9,pos9,0,0,0,rng)[-1] for _ in range(N9)])-tgt9,axis=1))*100)
    rng=np.random.default_rng(42)
    sds_cv.append(np.std(np.linalg.norm(np.array(
        [sim_delay_p(q9[0],tau9,pos9,...,...,...,rng)[-1] for _ in range(N9)])-tgt9,axis=1))*100)  # YOUR CODE HERE
    rng=np.random.default_rng(42)
    sds_pv.append(np.std(np.linalg.norm(np.array(
        [sim_delay_p(q9[0],tau9,pos9,...,...,...,rng)[-1] for _ in range(N9)])-tgt9,axis=1))*100)  # YOUR CODE HERE
    print(f'  T={T*1000:.0f}ms: OL={sds_ol[-1]:.3f} CV={sds_cv[-1]:.3f} PV={sds_pv[-1]:.3f}')

Tms=np.array(T_vals)*1000
fig,ax=plt.subplots(figsize=(9,6))
ax.plot(Tms,sds_ol,'s--',color=RED,lw=2.5,ms=8,label='Feedforward frontier (open-loop)')
ax.plot(Tms,sds_cv,'o-',color=TEAL,lw=2.5,ms=8,
        label='Central: gaze tracks hand (Δ=150ms, σ=1mm)')
ax.plot(Tms,sds_pv,'^-',color=ORANGE,lw=2.5,ms=8,
        label='Peripheral: gaze on target (Δ=80ms, σ=5mm)')
better=np.minimum(sds_cv,sds_pv)
ax.fill_between(Tms,better,sds_ol,
    where=np.array(better)<np.array(sds_ol),alpha=0.1,color=TEAL,label='Visual benefit zone')
ax.axvspan(190,260,alpha=0.12,color='#F39C12')
ax.text(275,max(sds_ol)*0.95,'Keele & Posner\ncrossover (~250 ms)',
    fontsize=9,color='#F39C12',fontweight='bold',va='top')
ax.set_xlabel('Movement Time (ms)',fontsize=13)
ax.set_ylabel('Endpoint SD (cm)',fontsize=13)
ax.set_title('The Updated Frontier (Kp=8, P-only, 10 cm reach)',fontweight='bold')
ax.legend(fontsize=9,loc='upper right')
plt.tight_layout(); plt.show()
print('Two crossovers:')
print('  1. ~250 ms: visual correction begins to help')
print('  2. ~700 ms: central overtakes peripheral (precision > speed)')


---
## Summary

1. **The Sensor class** encapsulates delay + noise — your implementation matches `smc.Sensor` exactly (Part 1)
2. **Sensory delay destabilizes PD feedback** — the same gain that corrects at Δ=0 causes oscillation at Δ=200 ms (Part 2)
3. **Peripheral (gaze on target) outperforms central (gaze tracks hand)** for continuous trajectory correction — speed beats precision (Part 3)
4. **Two crossovers:** visual correction helps for T > ~250 ms; central overtakes peripheral for T > ~700 ms (Part 4)

**What's missing:** P-only correction, no forward model, no proprioception, no sensory fusion.

**Next week:** Proprioception, the forward model, and the Kalman filter.
